# 01 Data Preprocessing — SENTINEL
Output for next phase: `artifacts/train.csv`, `valid.csv`, `test.csv` + schema report. Chronological split prevents leakage.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))


In [ ]:
import pandas as pd
from src.data_loader import load_labeled, load_profiles, schema_report
from src.preprocessing import preprocess, chronological_split
from src import config


In [ ]:
df = load_labeled()
print(schema_report(df))
print(df['is_suspicious'].value_counts(normalize=True))
print(df['typology'].value_counts())
print(df['date'].min(), df['date'].max())
prof = load_profiles()
print(prof['holder_profile'].value_counts())
# NOTE: account_profiles has post-hoc aggregates -> EDA only, never join as features

In [ ]:
df = preprocess(df)
assert df['transaction_id'].is_unique, 'dup IDs'
assert df['date'].is_monotonic_increasing
train, valid, test = chronological_split(df)
print(len(train), len(valid), len(test))
print(train['is_suspicious'].mean(), valid['is_suspicious'].mean(), test['is_suspicious'].mean())

In [ ]:
train.to_csv(config.ARTIFACTS_DIR/'train.csv', index=False)
valid.to_csv(config.ARTIFACTS_DIR/'valid.csv', index=False)
test.to_csv(config.ARTIFACTS_DIR/'test.csv', index=False)
print('saved to artifacts/ -> used by 02_feature_engineering.ipynb')